# mantiq: answers from your documents, with sources and checks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-Ousama/mantiq/blob/main/examples/mantiq-demo.ipynb)

[mantiq](https://github.com/Ahmed-Ousama/mantiq) reads your documents into the model's KV cache once, then answers any number of questions from them, without a retrieval pipeline. This notebook walks through the library with Qwen3-4B: answers, citations, answer checks, a comparison with RAG, handing questions the documents don't answer to a person, conversations and live updates. It takes about 15 minutes, most of it downloading the model and thinking over the answers the check isn't sure of.

**Before you start:** Runtime → Change runtime type → T4 GPU. On a GPU, the model loads in 4-bit automatically.

[GitHub](https://github.com/Ahmed-Ousama/mantiq) · [Paper (PDF)](https://github.com/Ahmed-Ousama/mantiq/releases/download/v0.2.0/mantiq-paper.pdf)

In [1]:
!pip install -q mantiq sentence-transformers

## Load the model

Qwen3 thinks before it answers by default; with thinking off, answers are short and fast. mantiq has built-in citation heads and an answer check for this model.

In [2]:
from mantiq import NaiveCAG

cag = NaiveCAG("Qwen/Qwen3-4B", chat_template_kwargs={"enable_thinking": False})

## Read the knowledge once

The knowledge is a few documents from an electronics store. mantiq runs the model over them once and keeps the KV cache. The document that changes most often, this week's offer, comes last, so updating it later is cheap.

In [3]:
documents = {
    "tv.md": "The Vision 55 is a 55-inch 4K TV with HDR10 and smart apps. It costs $499 and comes with a 2-year warranty.",
    "fridge.md": "The FrostFree is a 450-liter refrigerator with an A++ energy rating. It costs $899, with a 5-year warranty on the compressor.",
    "washer.md": "The CleanWave is a 9 kg front-load washing machine whose wash cycles adapt to the load. It costs $649.",
    "delivery.md": "Delivery takes 2 to 4 business days. It is free for orders over $300, otherwise $25.",
    "returns.md": "Returns are accepted within 14 days with the receipt. Refunds go to the original payment method.",
    "offers.md": "This week: 10% off the CleanWave washer.",
}
instructions = dict(
    prompt_instruction="You are a sales agent for an electronics store.",
    answer_instruction="Answer in one or two sentences, using only the given context.",
)

kv_cache, read_seconds = cag.prepare_kvcache_from_loader(documents, **instructions)
print(f"Read {kv_cache.get_seq_length()} tokens of knowledge in {read_seconds:.1f} s, once.")

Read 199 tokens of knowledge in 1.4 s, once.


## Ask questions

Each question reads only its own few tokens on top of the cache, so the first word comes right away. The rest of the answer then takes as long as the model needs to write it, about 9 tokens a second on a free T4.

In [4]:
import time

for question in ["I want a 4K screen, what do you have?", "How long does delivery take?", "Is there a deal on washing machines?"]:
    start, first = time.perf_counter(), None
    print(f"Q: {question}\nA: ", end="")
    for chunk in cag.stream_response(question, kv_cache, max_new_tokens=80):
        first = first or time.perf_counter() - start
        print(chunk, end="", flush=True)
    print(f"\n   (first word after {first:.2f} s, full answer after {time.perf_counter() - start:.1f} s)\n")

Q: I want a 4K screen, what do you have?
A: We have the Vision 55, a 55-inch 4K TV with HDR10 and smart apps, priced at $499 with a 2-year warranty.
   (first word after 0.16 s, full answer after 4.3 s)

Q: How long does delivery take?
A: Delivery takes 2 to 4 business days.
   (first word after 0.15 s, full answer after 1.2 s)

Q: Is there a deal on washing machines?
A: Yes, there is a 10% off deal on the CleanWave washing machine this week.
   (first word after 0.13 s, full answer after 2.3 s)



## Where does an answer come from?

`cite()` finds the document an answer draws on, in one pass: a few attention heads in the model look at the source, and mantiq knows which ones for 11 models. `supported` is low when the answer doesn't come from the knowledge at all, as for the last question.

In [5]:
for question in ["Can I send back a TV I don't like?", "Which fridge do you have, and is it efficient?", "Do you sell gaming consoles?"]:
    answer = cag.generate_response(question, kv_cache, max_new_tokens=60).strip()
    cited = cag.cite(question, answer, kv_cache, top_k=1)
    print(f"Q: {question}\nA: {answer}\n   source: {cited.citations[0].document}   supported: {cited.supported:.2f}\n")

Q: Can I send back a TV I don't like?
A: Yes, you can return the TV within 14 days of purchase with a receipt.
   source: returns.md   supported: 1.00

Q: Which fridge do you have, and is it efficient?
A: I have the FrostFree refrigerator, which is efficient due to its A++ energy rating.
   source: fridge.md   supported: 1.00

Q: Do you sell gaming consoles?
A: No, we don't sell gaming consoles, but we do sell the Vision 55 TV, which is great for gaming with its 4K resolution and HDR10 support.
   source: tv.md   supported: 0.02



## Is the answer right?

Most wrong answers are not made up from nothing: the model takes a fact from the wrong document, or slips on a step. `confidence` is meant for those. A small check reads the model's internal state as it answers; it was trained without reference answers, to predict whether the model's own thinking would agree. `generate_checked_response()` answers fast, and thinks it over only when the confidence is below 0.75. The check isn't perfect: it also sends some right answers to thinking, which only costs time.

In [6]:
questions = [
    "How long is the TV's warranty?",
    "Which product has the longest warranty?",
    "Does the washer come with a warranty?",
    "Which is cheaper this week, the washer or the TV?",
    "What do the TV and the washer cost together this week, with delivery?",
    "If I only buy a $200 item, how much do I pay with delivery?",
]
for question in questions:
    fast = cag.generate_response(question, kv_cache, max_new_tokens=80).strip()  # the answer the check looks at
    checked = cag.generate_checked_response(question, kv_cache, max_new_tokens=80)
    print(f"Q: {question}\nFast answer: {fast}\n   confidence: {checked.confidence:.2f}")
    if checked.rethought:
        print(f"After thinking: {checked.answer}")
    print()

Q: How long is the TV's warranty?
Fast answer: The Vision 55 TV comes with a 2-year warranty.
   confidence: 0.85

Q: Which product has the longest warranty?
Fast answer: The FrostFree refrigerator has the longest warranty, with a 5-year warranty on the compressor.
   confidence: 0.86

Q: Does the washer come with a warranty?
Fast answer: Yes, the CleanWave washing machine comes with a 2-year warranty.
   confidence: 0.42
After thinking: The CleanWave washing machine does not come with a specified warranty in the provided context.

Q: Which is cheaper this week, the washer or the TV?
Fast answer: This week, the CleanWave washer is cheaper at $584.50, while the Vision 55 TV is $499. The washer is discounted by 10%, making it the cheaper option.
   confidence: 0.42
After thinking: This week, the TV costs $499, while the washer is discounted to $584.10, making the TV cheaper.

Q: What do the TV and the washer cost together this week, with delivery?
Fast answer: The Vision 55 TV costs $499

## Compared with RAG

RAG retrieves the documents closest to each question and reads only those. Here is a small RAG over the same documents, with the same model and the same checks: an embedding model picks 2 of the 6 documents for each question. (Real knowledge bases are split into hundreds of chunks, and RAG reads a handful of them.) The only difference is what the model gets to see.

In [7]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
names = list(documents)
vectors = embedder.encode(list(documents.values()), normalize_embeddings=True)


def rag_cache(question, k=2):
    """Find the k documents closest to the question, and read only those."""
    scores = vectors @ embedder.encode(question, normalize_embeddings=True)
    found = [names[i] for i in scores.argsort()[::-1][:k]]
    cache, _ = cag.prepare_kvcache_from_loader({name: documents[name] for name in found}, **instructions)
    return found, cache


def first_word_after(start, cache, question):
    stream = cag.stream_response(question, cache)
    next(stream)
    stream.close()
    return time.perf_counter() - start


for question in ["How long does delivery take?", "Which is cheaper this week, the washer or the TV?",
                 "What do the TV and the washer cost together this week, with delivery?"]:
    start = time.perf_counter()
    found, rag_kv = rag_cache(question)
    rag_first = first_word_after(start, rag_kv, question)
    mantiq_first = first_word_after(time.perf_counter(), kv_cache, question)
    print(f"Q: {question}")
    print(f"RAG    (found {', '.join(found)}; first word after {rag_first:.2f} s)")
    print(f"       {cag.generate_checked_response(question, rag_kv, max_new_tokens=80).answer}")
    print(f"mantiq (sees all 6 documents; first word after {mantiq_first:.2f} s)")
    print(f"       {cag.generate_checked_response(question, kv_cache, max_new_tokens=80).answer}\n")

Q: How long does delivery take?
RAG    (found delivery.md, returns.md; first word after 0.34 s)
       Delivery takes 2 to 4 business days.
mantiq (sees all 6 documents; first word after 0.13 s)
       Delivery takes 2 to 4 business days.

Q: Which is cheaper this week, the washer or the TV?
RAG    (found offers.md, washer.md; first word after 0.30 s)
       This week, the CleanWave washer is discounted by 10%, costing $649, but the TV's price and any discounts are not provided, so we cannot determine which is cheaper.
mantiq (sees all 6 documents; first word after 0.12 s)
       This week, the TV costs $499, while the washer (with 10% off) costs $584.10, making the TV cheaper.

Q: What do the TV and the washer cost together this week, with delivery?
RAG    (found offers.md, washer.md; first word after 0.30 s)
       The context provided does not include the price of the TV or delivery costs, so I cannot calculate the total. Please check with the store for specific details.
mantiq (see

RAG is only as good as what it finds. Here it didn't find the TV, so it couldn't answer the last two questions; when a model doesn't notice such a gap, it makes the missing fact up. mantiq reads every document for every question, and its first word still comes sooner: RAG has to embed each question and read the documents it found before it can answer.

## When the documents don't have the answer

A sales agent shouldn't make things up. Give `generate_checked_response()` a `fallback`, and questions the documents don't answer go to a person instead. A small check reads the model's internal state as it reads the question, in one short pass: when the documents clearly don't answer it, you get the fallback right away, and `declined` is `True`, so your app can pass the question on. When that isn't clear, the model answers, and the fallback replaces the answer if the documents don't state it. Below, each declined question also shows what the model says without a fallback.

In [8]:
fallback = "I'm not sure about that, so I'll pass your question to a colleague."

for question in [
    "How long is the TV's warranty?",
    "Can I get the washer installed at home?",
    "Do you sell gaming consoles?",
    "Can I pay in installments?",
    "Does the TV support Dolby Vision?",
    "If I only buy a $200 item, how much do I pay with delivery?",
]:
    checked = cag.generate_checked_response(question, kv_cache, max_new_tokens=80, fallback=fallback)
    print(f"Q: {question}\nA: {checked.answer}")
    if checked.declined:  # here your app would hand the question to a person
        print(f"   (declined; without a fallback: {cag.generate_response(question, kv_cache, max_new_tokens=60).strip()})")
    print()

Q: How long is the TV's warranty?
A: The Vision 55 TV comes with a 2-year warranty.

Q: Can I get the washer installed at home?
A: I'm not sure about that, so I'll pass your question to a colleague.
   (declined; without a fallback: No, the CleanWave washing machine does not include home installation services. You would need to arrange for installation separately.)

Q: Do you sell gaming consoles?
A: I'm not sure about that, so I'll pass your question to a colleague.
   (declined; without a fallback: No, we don't sell gaming consoles, but we do sell the Vision 55 TV, which is great for gaming with its 4K resolution and HDR10 support.)

Q: Can I pay in installments?
A: I'm not sure about that, so I'll pass your question to a colleague.
   (declined; without a fallback: Based on the information provided, there is no mention of installment payment options available for the products listed.)

Q: Does the TV support Dolby Vision?
A: I'm not sure about that, so I'll pass your question to a c

## Conversations

By default each question stands alone. `chat()` keeps the earlier turns, so a follow-up like "how much is it?" works. The knowledge cache is shared, and each follow-up reads only what is new.

In [9]:
chat = cag.chat(kv_cache)
for question in ["Do you have a 4K TV?", "How much is it?", "And how long is its warranty?"]:
    print(f"Q: {question}\nA: {chat.ask(question, max_new_tokens=60)}\n")

Q: Do you have a 4K TV?
A: Yes, we have the Vision 55, a 55-inch 4K TV with HDR10 and smart apps, priced at $499 with a 2-year warranty.

Q: How much is it?
A: The Vision 55 is $499 with a 2-year warranty.

Q: And how long is its warranty?
A: The Vision 55 comes with a 2-year warranty.



## Update the knowledge

When a document changes, update the cache instead of building it again. mantiq re-reads the knowledge only from the first change onward, so changing the offer at the end costs a fraction of reading everything.

In [10]:
documents["offers.md"] = "This week: the Vision 55 TV is $399 instead of $499."
kv_cache, seconds = cag.update_kvcache(kv_cache, documents)
print(f"Updated in {seconds:.2f} s (reading everything took {read_seconds:.2f} s)\n")

question = "Is there a deal on TVs this week?"
print(f"Q: {question}\nA: {cag.generate_response(question, kv_cache, max_new_tokens=60).strip()}")

Updated in 0.11 s (reading everything took 1.43 s)

Q: Is there a deal on TVs this week?
A: Yes, the Vision 55 TV is on sale this week for $399 instead of the regular price of $499.


## Use your own documents

- `pip install "mantiq[loaders]"`, then `LoaderFactory.get_loader("PDF", "catalog.pdf")` reads PDFs, web pages, CSV, JSON, Markdown and more.
- `prepare_kvcache_from_loader(..., saving_path="cache")` saves the cache, and `cag.read_kv_cache("cache/kv_cache.pt")` loads it in another session.
- Citation heads are built in for Qwen3, Llama 3.1 and 3.2, Gemma 3, Mistral 7B and Phi-4-mini; `cag.calibrate_citations(kv_cache)` finds them for another model in a few minutes.
- The check behind the `fallback` is built in for Qwen3-4B; with other models, the model is asked instead, so check how often it declines on a few of your own questions first. The answer check, and thinking over unsure answers, are built in for Qwen3-4B and Qwen3-1.7B.

More in the [README](https://github.com/Ahmed-Ousama/mantiq#readme). Questions and ideas are welcome in the [issues](https://github.com/Ahmed-Ousama/mantiq/issues).